# Answers 02 Force densities

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**Given the force densities in the bars, where is ${\color{#d62828} n_0}$?**

Fixed nodes:
$$ \color{#2a6fdb}{n_1 \, (0, 0, 0) \text{ m}} $$
$$ \color{#2a6fdb}{n_2 \, (5, 0, 3) \text{ m}} $$
$$ \color{#2a6fdb}{n_3 \, (0, 7, 3) \text{ m}} $$
$$ \color{#2a6fdb}{n_4 \, (7, 5, 0) \text{ m}} $$

Load on the free node:
$$ \color{#2e9e4a}{p \, (0, 0, -5) \text{ kN}} $$

The free node ${\color{#d62828} n_0}$ is connected to four fixed nodes ${\color{#2a6fdb} n_1 \ldots n_4}$ by bars $e_0, e_1, e_2, e_3$:

$$
\newcommand{\cell}[1]{\hphantom{-}\mathclap{#1}\hphantom{-}}
\begin{array}{rcc}
 & & \begin{matrix} \cell{n_0} & \cell{n_1} & \cell{n_2} & \cell{n_3} & \cell{n_4} \end{matrix} \\[2pt]
C = & \begin{matrix} e_0 \\ e_1 \\ e_2 \\ e_3 \end{matrix} &
\left[ \;
\begin{matrix}
\cell{1} & \cell{-1} & \cell{0} & \cell{0} & \cell{0} \\
\cell{1} & \cell{0} & \cell{-1} & \cell{0} & \cell{0} \\
\cell{1} & \cell{0} & \cell{0} & \cell{-1} & \cell{0} \\
\cell{1} & \cell{0} & \cell{0} & \cell{0} & \cell{-1}
\end{matrix}
\; \right]
\end{array}
$$

The force density:

$$ q = \frac{\text{force in the edge}}{\text{edge length}} = [1, 1, 1, 1] \text{ kN/m} $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_one_node.svg"
       style="width:100%; height:auto;"
       alt="Free node n0 with bars e0 to e3 to fixed nodes n1 to n4 and load p" />
  <p style="text-align:center;">
    <em>The free node n<sub>0</sub> connected by bars e<sub>0</sub> to e<sub>3</sub> to the fixed nodes n<sub>1</sub> to n<sub>4</sub>, with load p.</em>
  </p>
</div>

- What is the sum of forces in node ${\color{#d62828} n_0}$?
- What happens if the force density $q = 2$ in bars $e_1$ and $e_3$?

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">


# Network

In [1]:
import numpy as np

# connectivity: rows = bars e0..e3, columns = nodes n0..n4
C = np.array([
    [1, -1,  0,  0,  0],
    [1,  0, -1,  0,  0],
    [1,  0,  0, -1,  0],
    [1,  0,  0,  0, -1],
])

free  = [0]            # n0
fixed = [1, 2, 3, 4]   # n1..n4

C_N = C[:, free]
C_F = C[:, fixed]

# fixed node coordinates (n1..n4)
x_F = np.array([0, 5, 0, 7])
y_F = np.array([0, 0, 7, 5])
z_F = np.array([0, 3, 3, 0])

# loads on the free node
p_x = np.array([0])
p_y = np.array([0])
p_z = np.array([-5])

# force densities, one per bar
q = np.array([1, 1, 1, 1])
Q = np.diag(q)

# Solve

One linear solve per axis. No iterations.

$$
\begin{aligned}
x_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_x - C_N^{T}\,Q\,C_F\,x_F\right) \\
y_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_y - C_N^{T}\,Q\,C_F\,y_F\right) \\
z_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_z - C_N^{T}\,Q\,C_F\,z_F\right)
\end{aligned}
$$

In [2]:
# equation 6.32
A = C_N.T @ Q @ C_N
x_N = np.linalg.solve(A, p_x - C_N.T @ Q @ C_F @ x_F)
y_N = np.linalg.solve(A, p_y - C_N.T @ Q @ C_F @ y_F)
z_N = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_F)

print(x_N, y_N, z_N)   # [3.] [3.] [0.25]

[3.] [3.] [0.25]


# Sum of Forces

The force follows from force density times length, $f = q \, l$. Each bar pulls ${\color{#d62828} n_0}$ towards its fixed node:

$$ \sum_i q_i \,({\color{#2a6fdb} n_i} - {\color{#d62828} n_0}) + {\color{#2e9e4a} p} = 0 $$

In [3]:
# all coordinates, n0 first
x = np.concatenate([x_N, x_F])
y = np.concatenate([y_N, y_F])
z = np.concatenate([z_N, z_F])

# bar lengths and bar forces
l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)
f = q * l
print("bar lengths =", np.round(l, 2), "m")
print("bar forces  =", np.round(f, 2), "kN")

# pull of each bar on n0, split in x, y, z
pull_x = -q * (C @ x)
pull_y = -q * (C @ y)
pull_z = -q * (C @ z)

sum_x = pull_x.sum() + p_x[0]
sum_y = pull_y.sum() + p_y[0]
sum_z = pull_z.sum() + p_z[0]

print("sum of forces in n0 =", np.round([sum_x, sum_y, sum_z], 2), "kN")

bar lengths = [4.25 4.53 5.71 4.48] m
bar forces  = [4.25 4.53 5.71 4.48] kN
sum of forces in n0 = [0. 0. 0.] kN


The sum of forces is zero: ${\color{#d62828} n_0}$ is in equilibrium. The four bars together push up 5 kN and cancel the load.
The bar forces are not 1 kN. They follow from the lengths: long bar, big force.

# Change the Force Densities

$$ q = [1, 2, 1, 2] \text{ kN/m} $$

In [4]:
q2 = np.array([1, 2, 1, 2])
Q2 = np.diag(q2)

# equation 6.32
A2 = C_N.T @ Q2 @ C_N
x_N2 = np.linalg.solve(A2, p_x - C_N.T @ Q2 @ C_F @ x_F)
y_N2 = np.linalg.solve(A2, p_y - C_N.T @ Q2 @ C_F @ y_F)
z_N2 = np.linalg.solve(A2, p_z - C_N.T @ Q2 @ C_F @ z_F)

x2 = np.concatenate([x_N2, x_F])
y2 = np.concatenate([y_N2, y_F])
z2 = np.concatenate([z_N2, z_F])

l2 = np.sqrt((C @ x2)**2 + (C @ y2)**2 + (C @ z2)**2)
f2 = q2 * l2

print(x_N2, y_N2, z_N2)
print("bar lengths =", np.round(l2, 2), "m")
print("bar forces  =", np.round(f2, 2), "kN")
print("sum of forces in n0 =", np.round([p_x[0] - (q2 * (C @ x2)).sum(),
                                         p_y[0] - (q2 * (C @ y2)).sum(),
                                         p_z[0] - (q2 * (C @ z2)).sum()], 2), "kN")

[4.] [2.83333333] [0.66666667]
bar lengths = [4.95 3.8  6.23 3.76] m
bar forces  = [4.95 7.61 6.23 7.52] kN
sum of forces in n0 = [ 0. -0.  0.] kN


# Viewer

Gray: all $q = 1$. Black: $q = 2$ in bars $e_1$ and $e_3$. The free node is red, the fixed nodes are blue, the load is green.

In [5]:
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY  = Color(0.70, 0.70, 0.70, 1.0)

session = Session("fdm_exercise_2")

# the net with q = 1, then the net with q = 2 in bars e1 and e3
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), [GRAY] * 4, [2.0] * 4)
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x2, y2, z2]).tolist()), [Color.black()] * 4, [3.0] * 4)

# node colours: the session holds the nodes of the first net, then the nodes of the second net
for net in range(2):
    for k in range(5):
        point = session.objects.points[net * 5 + k]
        point.pointcolor = RED if k in free else BLUE
        point.width = 12.0

# the load on n0, drawn 1 m long
load = Line(x2[0], y2[0], z2[0], x2[0], y2[0], z2[0] - 1.0)
load.linecolor = GREEN
load.width = 3.0
load.arrowhead = Arrowhead.END
session.add_line(load)

show(session, "fdm_exercise_2")

# Observation

${\color{#d62828} n_0}$ moves towards ${\color{#2a6fdb} n_2}$ and ${\color{#2a6fdb} n_4}$: bars $e_1$ and $e_3$ pull harder, so they get shorter. The node also rises: more force density carries the same load with less sag.
The sum of forces is still zero. Any choice of positive $q$ gives a net in equilibrium. Changing $q$ is how you design the shape.